In [1]:
# Código para extraer el json de:
# https://datos.madrid.es/egob/catalogo/206974-0-agenda-eventos-culturales-100.json

# 

# Extracción imágenes:
Funciones para extraer imagen de la página de detalle de cada actividad

In [2]:
# %pip install firebase-admin python-dotenv pyrebase4 fake-useragent
# %pip install fake-useragent

In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin  
import json
from datetime import datetime
import os
import pyrebase
from dotenv import load_dotenv
from fake_useragent import UserAgent
import random


In [4]:
def extraer_imagen(URL, ID_O_CLASE_DEL_DIV, URL_BASE):
    # 2. Descargar el HTML de la página
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Referer": "https://google.com" # Hace parecer que vienes desde Google
    }
    

    ua = UserAgent()

    # En tu bucle de peticiones:
    headers = {
        "User-Agent": ua.random,
        "Accept-Language": "es-ES,es;q=0.9"
    }
    respuesta = requests.get(URL, headers=headers)

    if respuesta.status_code != 200:
        print(f"Error al acceder a la página: {respuesta.status_code}")
        return None

    # 3. Parsear el HTML
    soup = BeautifulSoup(respuesta.text, "html.parser")

    # 4. Buscar el div específico (Elige Opción A o Opción B)
    # Opción A: Si el div tiene una clase CSS específica
    contenedor = soup.find("div", class_=ID_O_CLASE_DEL_DIV)

    # Opción B: Si el div tiene un ID único (descomenta la línea de abajo)
    # contenedor = soup.find('div', id=ID_O_CLASE_DEL_DIV)

    if not contenedor:
        # print("No se encontró el div especificado.")
        return None

    # 5. Extraer la etiqueta de la imagen dentro de ese div
    etiqueta_img = contenedor.find("img")

    if not etiqueta_img:
        print("No se encontró ninguna imagen dentro del div.")
        return None

    # 6. Obtener la URL de la imagen (atributo 'src')
    url_imagen = etiqueta_img.get("src")

    # Asegurar que la URL sea completa si es relativa
    if url_imagen and not url_imagen.startswith(("http://", "https://")):
        url_imagen = urljoin(URL_BASE, url_imagen)

    print(f"URL de la imagen encontrada: {url_imagen}")
    return url_imagen

In [5]:
# EJEMPLO: 

# 1. Configuración de la URL y el contenedor
URL = "https://www.madrid.es/sites/v/index.jsp?vgnextchannel=ca9671ee4a9eb410VgnVCM100000171f5a0aRCRD&vgnextoid=0b6fae21a7cbb910VgnVCM100000891ecb1aRCRD"
ID_O_CLASE_DEL_DIV = "image-content ic-right"  # Cambia esto según el ID o clase del div que contiene la imagen
URL_BASE = "https://www.madrid.es"  # Cambia esto si la URL base es diferente

# Ejecutar la función
imagen_actividad  = extraer_imagen(URL, ID_O_CLASE_DEL_DIV, URL_BASE)
# imagen_actividad


Error al acceder a la página: 403


# Procesamiento actividades:
Funciones para extraer el listado de actividades, con su título, fecha, enlace a la página de detalle, etc.

In [6]:
# PROCESAMIENTO COMPLETO DE ACTIVIDADES (equivalente a app.js)

from datetime import datetime
import math

# ==========================================
# FUNCIONES DE EXTRACCIÓN (de app.js)
# ==========================================

def extract_category(type_str):
    """Extrae categoría legible del tipo de evento (equivalente a extractCategory en app.js)"""
    if not type_str:
        return 'Otras'
    
    type_lower = str(type_str).lower()
    
    if 'cine' in type_lower or 'film' in type_lower:
        return 'Cine'
    if 'teatro' in type_lower or 'theatre' in type_lower:
        return 'Teatro'
    if 'musica' in type_lower or 'music' in type_lower or 'concierto' in type_lower:
        return 'Música'
    if 'exposicion' in type_lower or 'exhibition' in type_lower:
        return 'Exposiciones'
    if 'taller' in type_lower or 'workshop' in type_lower:
        return 'Talleres'
    if 'deporte' in type_lower or 'sport' in type_lower:
        return 'Deportes'
    if 'danza' in type_lower or 'dance' in type_lower:
        return 'Danza'
    if 'literatura' in type_lower or 'book' in type_lower:
        return 'Literatura'
    if 'infantil' in type_lower or 'cuentacuentos' in type_lower or 'circo' in type_lower:
        return 'Infantil'
    if 'fiesta' in type_lower or 'sanisidro' in type_lower:
        return 'Fiestas'
    if 'destacada' in type_lower:
        return 'Destacada'
    if 'conferencia' in type_lower:
        return 'Conferencias'
    if 'excursion' in type_lower or 'visita' in type_lower:
        return 'Excursiones'
    if 'campamento' in type_lower:
        return 'Campamentos'
    
    return 'Otras'


def extract_district(district_url):
    """Extrae nombre del distrito desde URL (equivalente a extractDistrict en app.js)"""
    if not district_url:
        return 'Desconocido'
    
    parts = str(district_url).split('/')
    name = parts[-1] if parts else ''
    
    district_map = {
        'Fuencarral-ElPardo': 'Fuencarral - El Pardo',
        'CiudadLineal': 'Ciudad Lineal',
        'PuenteDeVallecas': 'Puente de Vallecas',
        'Moncloa-Aravaca': 'Moncloa - Aravaca',
        'SanBlas-Canillejas': 'San Blas-Canillejas',
        'VillaDeVallecas': 'Villa de Vallecas'
    }
    
    return district_map.get(name, name)


# ==========================================
# FUNCIONES DE CÁLCULO (de app.js)
# ==========================================

def extract_hour(time_str):
    """Extrae la hora de un string de tiempo (equivalente a extractHour en app.js)"""
    if not time_str:
        return None
    try:
        parts = str(time_str).split(':')
        if len(parts) >= 2:
            return int(parts[0])
    except (ValueError, TypeError):
        return None
    return None


# ==========================================
# FUNCIÓN PRINCIPAL DE PROCESAMIENTO
# ==========================================

def process_activities(raw_data, user_location=None):
    """
    Procesa las actividades igual que hace app.js:
    1. Extrae y transforma los datos
    2. Filtra actividades pasadas
    3. Normaliza categorías y distritos
    4. Añade campos calculados (distance si hay ubicación)
    """
    graph = raw_data.get('@graph', [])
    today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    
    processed = []
    
    for item in graph:
        # print(f"Procesando actividad: {item.get('title', 'Sin título')}")
        # Extraer dirección y ubicación con seguridad
        address = item.get('address', {}) or {}
        district_obj = address.get('district', {}) or {}
        area = address.get('area', {}) or {}
        location_obj = item.get('location', {}) or {}
        
        # Procesar fechas
        dtstart = item.get('dtstart')
        date_val = None
        if dtstart:
            try:
                date_val = datetime.fromisoformat(dtstart.replace('Z', '+00:00').replace('+00:00', ''))
            except:
                pass
        
        dtend = item.get('dtend')
        end_date_val = None
        if dtend:
            try:
                end_date_val = datetime.fromisoformat(dtend.replace('Z', '+00:00').replace('+00:00', ''))
            except:
                pass
        
        # Calcular diferencia de días (si hay fecha de inicio y fin)
        if date_val and end_date_val:
            date_diff = (end_date_val - date_val).days
        else:
            date_diff = None

        # Clasificamos la duración del evento entre "Single-day", "Multiday" (> 1 < 7) o "Long-event" (>= 7):
        if date_diff is not None:
            if date_diff > 7:
                duration = "Larga duración (> 7 días)"
            elif date_diff > 1:
                duration = "Multi-día (2-7 días)"
            else:
                duration = "1 día"
        else:
            duration = "1 día"

        # Procesar coordenadas
        try:
            lat = float(location_obj.get('latitude')) if location_obj.get('latitude') else None
            lon = float(location_obj.get('longitude')) if location_obj.get('longitude') else None
        except (ValueError, TypeError):
            lat, lon = None, None

        # ID app (a partir de https://datos.madrid.es/egob/catalogo/tipo/evento/):
        app_id = item.get('@id', '')
        app_id = app_id.split('/')[-1] if app_id else str(random.random())

        # Construir actividad procesada
        activity = {
            'app_id': app_id,
            'id': item.get('@id', ''),
            'title': item.get('title', 'Sin título'),
            'description': item.get('description', ''),
            'category': extract_category(item.get('@type')),
            'location': item.get('event-location', ''),
            'district': extract_district(district_obj.get('@id')),
            'lat': lat,
            'lon': lon,
            'date': date_val.isoformat() if date_val else None,
            'endDate': end_date_val.isoformat() if end_date_val else None,
            'time': item.get('time', ''),
            'free': item.get('free') == 1 or item.get('free') is True,
            'price': item.get('price', ''),
            'audience': item.get('audience', ''),
            'link': item.get('link', ''),
            'street': area.get('street-address', '') if area else '',
            'date_diff': date_diff,
            'duration': duration
        }
        
        processed.append(activity)
    
    # Filtrar actividades pasadas (igual que en app.js)
    # filtered = [
    #     a for a in processed
    #     if a['date'] is None or datetime.fromisoformat(a['date']) >= today
    # ]
    filtered = [a for a in processed]
    
    return filtered

In [7]:

# ==========================================
# EJECUCIÓN DEL PROCESAMIENTO
# ==========================================

# Extraer json de ruta web: https://datos.madrid.es/egob/catalogo/206974-0-agenda-eventos-culturales-100.json
raw_data = json.loads(requests.get("https://datos.madrid.es/egob/catalogo/206974-0-agenda-eventos-culturales-100.json").text)

# Cargar datos crudos del archivo descargado
# with open('actividades.json', 'r', encoding='utf-8') as f:
#     raw_data = json.load(f)

# Procesar actividades (sin ubicación de usuario por defecto)
# Si quieres calcular distancias, pasa: user_location={'lat': 40.4168, 'lon': -3.7038}
all_activities_processed = process_activities(raw_data)

print(f"Total actividades procesadas: {len(all_activities_processed)}")

# Guardar actividades procesadas
with open('./data/actividades_procesadas.json', 'w', encoding='utf-8') as f:
    json.dump(all_activities_processed, f, ensure_ascii=False, indent=2)

print("✅ Actividades procesadas y guardadas en 'actividades_procesadas.json'")

# Mostrar ejemplo de actividad procesada
# if all_activities_processed:
#     print("\n📋 Ejemplo de actividad procesada:")
#     print(json.dumps(all_activities_processed[0], indent=2, ensure_ascii=False))


Total actividades procesadas: 1007
✅ Actividades procesadas y guardadas en 'actividades_procesadas.json'


# Procesamiento imagenes actividades:

In [43]:
# Pasos: 
# - Cargamos el fichero de actividades_procesadas.json
# - Cargamos el fichero de imagenes.json en un diccionario (app_id -> url_imagen)
# - Para cada actividad, obtenemos buscamos por app_id en el fichero de imagenes.json
# - Si existe el registro, pasamos al siguiente. 
# - Sino, lanzamos el script de extracción de link de imagen y lo añadimos al diciconario.




In [44]:
json_procesado = './data/actividades_procesadas.json'
json_imagenes = './data/imagenes.json'
# 1. Configuración de la URL y el contenedor
div_class = "image-content ic-right"  # Cambia esto según el ID o clase del div que contiene la imagen
url_base = "https://www.madrid.es"  # Cambia esto si la URL base es diferente
url_imagenes = "https://firebasestorage.googleapis.com/v0/b/actividades-madrid-2dbb6.firebasestorage.app/o/imagenes.json?alt=media"


In [45]:
import time
def load_json(path, default):
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            print(f"No se pudo leer {path}: {e}")
            return default
    return default


def save_json_atomic(path, data):
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

In [51]:
actividades = load_json(json_procesado, [])
if not isinstance(actividades, list):
    print(f"Formato inesperado en {json_procesado}: se esperaba una lista.")
    # return None

# imagenes_map = load_json(json_imagenes, {})
# if not isinstance(imagenes_map, dict):
#     print(f"Formato inesperado en {json_imagenes}: se esperaba un diccionario. Se re-inicializa.")
#     imagenes_map = {}
try:
    imagenes_map = json.loads(requests.get(url_imagenes).text)
except Exception as e:
    imagenes_map = {}

obtained = 0
total_new = 0

for activity in actividades:

    app_id = activity.get('app_id')
    if not app_id:
        continue

    # Skip if already has an entry (even if null)
    if app_id in imagenes_map:
        # print("skip")
        continue
    print(app_id)
    link = activity.get('link')
    print(link)
    if not link:
        imagenes_map.setdefault(app_id, None)
        continue

    # print(f"Extrayendo imagen para app_id={app_id} desde {link}")
    try:
        img = extraer_imagen(link, div_class, url_base)
    except Exception as e:
        # print(f"Error extrayendo imagen para {app_id}: {e}")
        img = None

    imagenes_map[app_id] = img
    total_new += 1
    if img:
        obtained += 1
        # print(f" -> imagen encontrada: {img}")
    else:
        # print(" -> no se encontró imagen")
        obtained += 0

    # Guardar incrementalmente para evitar perder trabajo
    try:
        save_json_atomic(json_imagenes, imagenes_map)
    except Exception as e:
        print(f"No se pudo guardar {json_imagenes}: {e}")

    # Parar si hemos obtenido el máximo de imágenes requeridas
    if total_new >= 5000:
        print(f"Se han extraído {obtained} imágenes. Deteniendo.")
        break

    time.sleep(random.uniform(5, 7)) 

print(f"Hecho. Imágenes obtenidas en esta ejecución: {obtained}. Entradas nuevas procesadas: {total_new}.")


50321933-anatomia-desgaste.json
http://www.madrid.es/sites/v/index.jsp?vgnextchannel=ca9671ee4a9eb410VgnVCM100000171f5a0aRCRD&vgnextoid=6914ebf1ba46e910VgnVCM200000f921e388RCRD
Error al acceder a la página: 403
50173939-arte-botanico-taller-estampado-vegetal.json
http://www.madrid.es/sites/v/index.jsp?vgnextchannel=ca9671ee4a9eb410VgnVCM100000171f5a0aRCRD&vgnextoid=f8ebef7e4a54b910VgnVCM200000f921e388RCRD
Error al acceder a la página: 403


KeyboardInterrupt: 

In [48]:
link

'http://www.madrid.es/sites/v/index.jsp?vgnextchannel=ca9671ee4a9eb410VgnVCM100000171f5a0aRCRD&vgnextoid=0d4814a4c646e910VgnVCM200000f921e388RCRD'

# Firebase subida

In [ ]:

# 1. Cargar las variables de entorno
load_dotenv()

# 2. Estructurar la configuración mapeando tus variables
config = {
    "apiKey": os.getenv("FIREBASE_apiKey"),
    "authDomain": os.getenv("FIREBASE_authDomain"),
    "projectId": os.getenv("FIREBASE_projectId"),
    "storageBucket": os.getenv("FIREBASE_storageBucket"),
    "messagingSenderId": os.getenv("FIREBASE_messagingSenderId"),
    "appId": os.getenv("FIREBASE_appId"),
    "databaseURL": ""  # Se puede dejar vacío si solo usas Storage
}

# 3. Inicializar Firebase y el servicio de Storage
firebase = pyrebase.initialize_app(config)
storage = firebase.storage()

# 4. Definir rutas de los archivos
# Usar el archivo procesado (equivalente a app.js) en lugar del raw
ruta_archivo_local = "./data/actividades_procesadas.json"
ruta_destino_en_nube = "actividades_procesadas.json"

# 5. Subir el archivo actividades procesadas.json a Firebase Storage
# En pyrebase se usa child() para la ruta destino y put() para el archivo local
storage.child(ruta_destino_en_nube).put(ruta_archivo_local)

# 6. Confirmar que el archivo se ha subido correctamente
try:
    url = storage.child(ruta_destino_en_nube).get_url(None)
    print(f"Archivo subido con éxito. URL de acceso: {url}")
except Exception as e:
    print(f"Error al obtener la URL del archivo subido: {e}")
print("¡Archivo subido con éxito usando las credenciales del .env!")

# Subir también el fichero de imagenes (opcional)
ruta_imagenes_local = "./data/imagenes.json"
ruta_imagenes_destino = "imagenes.json"
storage.child(ruta_imagenes_destino).put(ruta_imagenes_local)
try:
    url_imagenes = storage.child(ruta_imagenes_destino).get_url(None)
    print(f"Archivo de imágenes subido con éxito. URL de acceso: {url_imagenes}")
except Exception as e:
    print(f"Error al obtener la URL del archivo de imágenes subido: {e}")

Archivo subido con éxito. URL de acceso: https://firebasestorage.googleapis.com/v0/b/actividades-madrid-2dbb6.firebasestorage.app/o/actividades_procesadas.json?alt=media
¡Archivo subido con éxito usando las credenciales del .env!
Archivo de imágenes subido con éxito. URL de acceso: https://firebasestorage.googleapis.com/v0/b/actividades-madrid-2dbb6.firebasestorage.app/o/imagenes.json?alt=media


In [ ]:
json.loads(requests.get(url_imagenes).text)  

{'12008124-10-festival-internacional-folclore-latina-folkfestival-ara-madrid-.json': 'https://www.madrid.es/UnidadesDescentralizadas/DistritoLatina/Actividades/Fiestas%20de%20Aluche/7_7_Ballet_Ara.jpg',
 '12366331-3devermu-funk-disco-djset-vinilo.json': None,
 '50257645-4-escenas-4-estilos.json': 'https://www.madrid.es/UnidadesDescentralizadas/CiudadDistrito/ficheros/CIUDAD_DISTRITO_233150_es.jpeg',
 '50149771-6-x-4-cueros.json': 'https://www.madrid.es/UnidadesDescentralizadas/DistritoRetiro/FICHEROS/FICHEROS%20ACTIVIDADES%20MAYO/30%20may%20CV%20FIGUE%202026_6%20x%204%20&%20CUEROS.jpg',
 '50297173-75-anos-direccion-artistica-cinematografica-gil-parrondo-creador-suenos.json': 'https://www.madrid.es/UnidadWeb/UGBBDD/Actividades/Distritos/Arganzuela/Eventos/ficheros/gilparrondo.png',
 '50319880-80-aniversario-revista-insula.json': 'https://www.madrid.es/UnidadesDescentralizadas/Bibliotecas/BibliotecasPublicas/Actividades/PresentacionesLibros_ActosLiterarios/ficheros/260605_Insula_260x260.

In [ ]:
url_imagenes

'https://firebasestorage.googleapis.com/v0/b/actividades-madrid-2dbb6.firebasestorage.app/o/imagenes.json?alt=media'

In [ ]:
ruta_local = "./data/actividades_orig.json"
ruta_destino = "actividades_orig.json"
storage.child(ruta_destino).put(ruta_local)
url_orig = storage.child(ruta_destino).get_url(None)
url_orig

'https://firebasestorage.googleapis.com/v0/b/actividades-madrid-2dbb6.firebasestorage.app/o/actividades_orig.json?alt=media'